In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

df = pd.read_csv("IBM.csv", index_col="Date", parse_dates=True)
df.head(3)
df[["Close"]].plot(figsize=(10,5),title="IBM stock price");

In [ ]:
train_data, test_data = df[0:int(len(df)*0.8)], df[int(len(df)*0.8):]
train_arima = train_data['Close']
test_arima = test_data['Close']
history = train_arima.to_list()
model_fit = ARIMA(history, order=(1,1,0)).fit()

In [ ]:
n = 5 # num of predictions
yhat = model_fit.forecast(steps=n)
print(yhat)
print(test_arima.iloc[0:n])

In [ ]:
predictions = list()
history = train_arima.to_list()
y = test_arima
for yy in y:
    model_fit = ARIMA(history, order=(1,1,0)).fit()
    yhat = model_fit.forecast()[0]
    predictions.append(yhat)
    history.append(yy)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(16,8))
plt.plot(df.index[-1600:], df['Close'].tail(1600), color='green', label = 'Train Stock Price')
plt.plot(test_data.index, y, color = 'red', label = 'Real Stock Price')
plt.plot(test_data.index, predictions, color = 'blue', label = 'Predicted Stock Price')
plt.title('IBM Stock Price Prediction')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
pip install pmdarima # must be installed first in Colab

In [ ]:
from pmdarima import auto_arima
stepwise_fit = auto_arima(history, start_p = 0, start_q = 0,seasonal = False, d = None, trace = True)
# To print the summary
print(stepwise_fit.summary())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

df = pd.read_csv('AirPassengers.csv',index_col ='Month',parse_dates = True)
print(df.head())

plt.plot(df['#Passengers'])
plt.show()

In [ ]:
result = seasonal_decompose(df['#Passengers'], model ='multiplicative')

plt.figure(figsize=(10,8))

plt.subplot(4, 1, 1)
plt.plot(df['#Passengers'], label='Airline passengers')
plt.legend()
plt.subplot(4, 1, 2)
plt.plot(result.trend, label='Trend')
plt.legend()
plt.subplot(4, 1, 3)
plt.plot(result.seasonal, label='Seasonal')
plt.legend()
plt.subplot(4, 1, 4)
plt.plot(result.resid, label='Residuals')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from pmdarima import auto_arima

# Fit auto_arima function to AirPassengers dataset

stepwise_fit = auto_arima(df['#Passengers'], start_p = 1, start_q = 1,max_p = 3, max_q = 3, m = 12,start_P = 0, seasonal = True,
d = None, D = 1, trace = True,
error_action ='ignore', 
suppress_warnings = True, 
stepwise = True) 

stepwise_fit.summary()

In [ ]:
train = df.iloc[:len(df)-12]
test = df.iloc[len(df)-12:]

from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(train['#Passengers'],order = (0, 1, 1),seasonal_order =(2, 1, 1, 12))

result = model.fit()

result.summary()

In [ ]:
start = len(train)
end = len(train) + len(test) - 1

# Predictions for one-year against the test set

predictions = result.predict(start, end,typ = 'levels').rename("Predictions")


predictions.plot(legend = True)
test['#Passengers'].plot(legend = True)

In [ ]:
from sklearn.metrics import mean_squared_error
from statsmodels.tools.eval_measures import rmse

rmse(test["#Passengers"], predictions)

In [ ]:
# Train the model on the FULL dataset

model = model = SARIMAX(df['#Passengers'],order = (0, 1, 1),seasonal_order =(2, 1, 1, 12))

result = model.fit()

# Forecast for the next 3 years
forecast = result.predict(start = len(df),end = (len(df)-1) + 3 * 12,typ = 'levels').rename('Forecast')


df['#Passengers'].plot(figsize = (12, 5), legend = True)
forecast.plot(legend = True)